# Delta System — Kaggle Training

**Data**: Wikipedia consecutive paragraph pairs (always available on HuggingFace)  
- A = paragraph N of an article  
- B = paragraph N + paragraph N+1  
- novel = paragraph N+1 (new information added to A)  

**Goal**: Train on 8000 pairs, evaluate on 1000 held-out unseen pairs.  
**Pass criteria (held-out only)**: `DELTA_PPL > 2` AND `SPECIFICITY > 2`

In [ ]:
# Cell 1 — Install
!pip install transformers scikit-learn datasets -q

In [ ]:
# Cell 2 — Setup paths
import os, sys
from pathlib import Path

REPO = Path('/kaggle/working/multihop-retrieval')
if REPO.exists():
    os.system(f'git -C {REPO} pull')   # always pull so new files are present
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {REPO}')

DELTA_DIR = REPO / 'delta_system'
sys.path.insert(0, str(DELTA_DIR))
print(f'delta_system: {DELTA_DIR}')
print(f'Files: {sorted([f.name for f in DELTA_DIR.glob("*.py")])}')

In [ ]:
# Cell 3 — Config
import torch

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
N_TRAIN   = 8000
N_EVAL    = 1000
STEPS     = 2000
BS        = 16
LR        = 1e-4
LAM_S     = 1.0
LAM_SPEC  = 1.0
MARGIN    = 2.0
LOG_EVERY = 200
MAX_LEN   = 128

print(f'Device : {DEVICE}')
print(f'Config : {N_TRAIN} train / {N_EVAL} held-out | {STEPS} steps | bs={BS}')

In [ ]:
# Cell 4 — Load data from Wikipedia (guaranteed available on HuggingFace)
# A = paragraph N, B = paragraph N + paragraph N+1, novel = paragraph N+1
# Consecutive paragraphs naturally introduce new information — perfect (A, B, novel) structure

from datasets import load_dataset

def load_wikipedia_pairs(n_train=8000, n_eval=1000):
    total  = n_train + n_eval
    pairs  = []

    print('Loading Wikipedia (streaming)...')
    ds = load_dataset(
        'wikimedia/wikipedia',
        '20231101.en',
        split='train',
        streaming=True,
    )

    for article in ds:
        text = article.get('text', '')
        if not text:
            continue

        # Split into paragraphs — Wikipedia uses double newline
        paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 80]

        # Create consecutive paragraph pairs from this article
        for i in range(len(paragraphs) - 1):
            A     = paragraphs[i]
            novel = paragraphs[i + 1]
            B     = A + ' ' + novel

            # Quality filters
            if len(A) < 80 or len(novel) < 80:
                continue
            if len(A) > 1500 or len(novel) > 1500:
                continue   # skip very long paragraphs (will truncate badly)

            pairs.append({'A': A, 'B': B, 'novel': novel})
            if len(pairs) >= total:
                break

        if len(pairs) >= total:
            break

        if len(pairs) % 1000 == 0 and len(pairs) > 0:
            print(f'  Collected {len(pairs)}/{total}...')

    print(f'Total pairs collected: {len(pairs)}')
    return pairs[:n_train], pairs[n_train:total]


train_pairs, eval_pairs = load_wikipedia_pairs(N_TRAIN, N_EVAL)
print(f'Train: {len(train_pairs)} | Eval (held-out): {len(eval_pairs)}')
print(f'\nExample A     : {train_pairs[0]["A"][:120]}')
print(f'Example novel : {train_pairs[0]["novel"][:120]}')

In [ ]:
# Cell 5 — Load model + tokenizer
from model import DeltaSystem
from transformers import BertTokenizerFast

tok   = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = DeltaSystem().to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params : {n_params/1e6:.1f}M')
print(f'BERT frozen      : {not next(model.bert.parameters()).requires_grad}')

In [ ]:
# Cell 6 — Training
import math
from torch.utils.data import DataLoader, Dataset
from losses import recon_loss, sparsity_loss, specificity_loss

class PairDS(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self):         return len(self.pairs)
    def __getitem__(self, i):  return self.pairs[i]['A'], self.pairs[i]['B']

def make_collate(tok):
    def collate(batch):
        A_texts = [x[0] for x in batch]
        B_texts = [x[1] for x in batch]
        eA = tok(A_texts, max_length=MAX_LEN, truncation=True,
                 padding='max_length', return_tensors='pt')
        eB = tok(B_texts, max_length=MAX_LEN, truncation=True,
                 padding='max_length', return_tensors='pt')
        return eA['input_ids'], eA['attention_mask'], eB['input_ids'], eB['attention_mask']
    return collate

dl  = DataLoader(PairDS(train_pairs), batch_size=BS, shuffle=True,
                 collate_fn=make_collate(tok), num_workers=2, pin_memory=True)
opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)

model.train()
step = 0
while step < STEPS:
    for batch in dl:
        if step >= STEPS: break
        A_ids, A_mask, B_ids, B_mask = [t.to(DEVICE) for t in batch]
        b = A_ids.size(0)

        logits, delta, delta_0, H_A, alpha = model(A_ids, A_mask, B_ids, B_mask)
        L_r    = recon_loss(logits, B_ids, B_mask)
        L_s    = sparsity_loss(delta, B_mask)
        L_spec = torch.tensor(0.0, device=DEVICE)
        if b > 1:
            idx_shift    = list(range(1, b)) + [0]
            logits_wrong = model.reconstruct(
                H_A, A_mask, delta[idx_shift], delta_0[idx_shift], B_ids, B_mask)
            L_spec = specificity_loss(logits, logits_wrong, B_ids, B_mask, margin=MARGIN)

        loss = L_r + LAM_S * L_s + LAM_SPEC * L_spec
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
        step += 1

        if step % LOG_EVERY == 0 or step == 1:
            ppl = math.exp(min(L_r.item(), 20))
            print(f'  step {step:4d}/{STEPS} | ppl={ppl:.1f} | L_spec={L_spec.item():.4f}')

print('Training complete.')

In [ ]:
# Cell 7 — Save checkpoint
ckpt_dir = Path('/kaggle/working/checkpoints')
ckpt_dir.mkdir(exist_ok=True)
trainable = {k: v for k, v in model.state_dict().items() if not k.startswith('bert.')}
torch.save(trainable, ckpt_dir / 'kaggle_model.pt')
print(f'Saved: {ckpt_dir / "kaggle_model.pt"}')

In [ ]:
# Cell 8 — HELD-OUT EVALUATION (unseen pairs — the key result)
from eval import evaluate

print('=' * 62)
print(f'HELD-OUT EVAL — {len(eval_pairs)} pairs never seen during training')
print('=' * 62)
held_out_results = evaluate(model, eval_pairs, tok)

In [ ]:
# Cell 9 — In-sample sanity check + final summary
print('=' * 62)
print('IN-SAMPLE CHECK — 200 training pairs')
print('=' * 62)
train_results = evaluate(model, train_pairs[:200], tok)

print('\n' + '=' * 62)
print('FINAL SUMMARY')
print('=' * 62)
print(f'  In-sample  DELTA_PPL : {train_results["delta_ppl"]:+.2f}')
print(f'  Held-out   DELTA_PPL : {held_out_results["delta_ppl"]:+.2f}  <- KEY')
print(f'  In-sample  SPEC      : {train_results["specificity"]:+.2f}')
print(f'  Held-out   SPEC      : {held_out_results["specificity"]:+.2f}')
print()
if held_out_results['pass']:
    print('  HELD-OUT PASS — system generalizes. Ready to scale further.')
else:
    print('  HELD-OUT FAIL — needs more data or steps.')
    if held_out_results['delta_ppl'] <= 2:
        print('    -> DELTA_PPL failed: increase STEPS or N_TRAIN')
    if held_out_results['specificity'] <= 2:
        print('    -> SPECIFICITY failed: increase LAM_SPEC or MARGIN')
print('=' * 62)

In [ ]:
# Cell 10 — Define and init DeltaDecoder
import os
from pathlib import Path

# Pull latest code (delta_decoder.py, embedding init fix)
repo_root = Path('/kaggle/working/multihop-retrieval')
if repo_root.exists():
    print('Pulling latest code...')
    os.system(f'git -C {repo_root} pull')
else:
    os.system(f'git clone https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git {repo_root}')

from delta_decoder import DeltaDecoder, train_decoder, show_examples

# Re-load trained G weights
ckpt_path = Path('/kaggle/working/checkpoints/kaggle_model.pt')
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt, strict=False)
    print(f'G weights loaded from {ckpt_path}')
else:
    print('No checkpoint found — using current model weights')

# Freeze G entirely
for p in model.parameters():
    p.requires_grad_(False)

# Create decoder + copy BERT embeddings into it
# WHY: default nn.Embedding uses N(0,1) → logit std ≈ 28 → softmax collapse → loss >> 20
# BERT embeddings have std ≈ 0.02, giving logit std ≈ 0.55 → trainable from step 1
dec_model = DeltaDecoder().to(DEVICE)
dec_model.copy_bert_embeddings(model.bert)
print('BERT embeddings copied into decoder word_emb + pos_emb')

n_dec = sum(p.numel() for p in dec_model.parameters() if p.requires_grad)
print(f'DeltaDecoder params : {n_dec/1e6:.2f}M')
print(f'G frozen            : {not any(p.requires_grad for p in model.parameters())}')

In [ ]:
# Cell 11 — Train DeltaDecoder
# G is frozen. Only the decoder learns.
# Starting from BERT embeddings so loss starts at ~10 nats (not 50+).
# Logging shows raw loss so we can confirm learning is happening.

DEC_STEPS     = 2000   # more steps now that init is correct
DEC_BS        = 16
DEC_LR        = 3e-4   # slightly higher than G training — decoder starts fresh
DEC_LOG_EVERY = 200

print('Training DeltaDecoder...')
print(f'  {DEC_STEPS} steps | bs={DEC_BS} | lr={DEC_LR}')
print(f'  Training on {len(train_pairs)} pairs')
print(f'  Expect loss to START near ~10 (not 50+) and DROP steadily')
print()

dec_model = train_decoder(
    g_model     = model,
    dec_model   = dec_model,
    train_pairs = train_pairs,
    tok         = tok,
    steps       = DEC_STEPS,
    bs          = DEC_BS,
    lr          = DEC_LR,
    log_every   = DEC_LOG_EVERY,
    device      = DEVICE,
)

# Save
dec_ckpt = Path('/kaggle/working/checkpoints/delta_decoder.pt')
torch.save(dec_model.state_dict(), dec_ckpt)
print(f'\nDecoder saved: {dec_ckpt}')

# Sanity check: δ_0 stats — should NOT all be zeros or identical
import numpy as np
model.eval()
with torch.no_grad():
    sample = train_pairs[:8]
    B_texts = [p['B'] for p in sample]
    enc = tok(B_texts, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
    B_ids, B_mask = enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE)
    A_texts = [p['A'] for p in sample]
    encA = tok(A_texts, max_length=128, truncation=True, padding='max_length', return_tensors='pt')
    A_ids, A_mask = encA['input_ids'].to(DEVICE), encA['attention_mask'].to(DEVICE)
    H_A = model._enc(A_ids, A_mask)
    H_B = model._enc(B_ids, B_mask)
    _, d0, _ = model.generate_delta(H_A, A_mask, H_B, B_mask)
print(f'\nδ_0 stats (should vary across examples):')
print(f'  mean abs: {d0.abs().mean().item():.4f}')
print(f'  std across examples: {d0.std(0).mean().item():.4f}')
print(f'  (if std~0, δ_0 is collapsed — bad)')

In [ ]:
# Cell 12 — Qualitative examples (the demo)
# For 10 held-out pairs:
#   - Compute δ_0 from frozen G
#   - Greedy-decode from DeltaDecoder using δ_0 as ONLY input (no A, no D_recon)
#   - Print: A snippet | True novel paragraph | What decoder generates
#
# What would make this result exciting:
#   - Decoder generates words/topics that appear in the true novel paragraph
#   - Decoder output is different for different pairs (not one generic sentence)
#   - Decoder does NOT just copy A (proves it uses δ_0, not background knowledge)
#
# What is normal / acceptable:
#   - Decoder won't reproduce the novel paragraph verbatim (it's a 2-layer decoder)
#   - Word-level match is not the goal — topic/entity match is what we look for
#   - Some examples will look off; 10 examples gives a fair qualitative picture

print('Running qualitative decoder demo on 10 HELD-OUT pairs...')
print('(δ_0 is the ONLY input — no A, no D_recon)')
print()

show_examples(
    g_model   = model,
    dec_model = dec_model,
    pairs     = eval_pairs,     # held-out pairs — never seen during G training
    tok       = tok,
    n         = 10,
    device    = DEVICE,
)

print()
print('Interpretation guide:')
print('  GOOD : decoded text mentions similar topics/entities as the true novel paragraph')
print('  GOOD : different examples produce clearly different decoded text')
print('  BAD  : all examples produce the same generic phrase (δ_0 collapsed)')
print('  BAD  : decoded text is identical to A text (no novelty captured)')